# Reproducing Sicilian–English–Italian NMT

One linear, narrated run of our whole model pipeline. Each step is a short explanation
(what & why, like the slides) followed by a cell that calls the implementation in the
repo's `.py` files — nothing is hidden in the notebook.

**Pipeline:** data → Standard-Sicilian preprocessing → NLLB-200 + LoRA → fine-tune →
evaluate. Implementations: `experiments/nllb/nllb_pipeline.py`,
`experiments/dataset/normalize_scn.py`. Run on a **GPU** runtime (Runtime → Change
runtime type → GPU).

## 0. Setup
Clone the repo (for the `.py` implementations), install deps, mount Drive (the prepared
dataset lives in `SicilianNMT-colab/data/`).

In [ ]:
!pip -q install transformers datasets peft sentencepiece sacrebleu accelerate
!pip -q uninstall -y torchao
import os, sys
if not os.path.exists('SicilianNMT'):
    !git clone -q https://github.com/dmeoli/SicilianNMT.git
sys.path += ['SicilianNMT/experiments/nllb', 'SicilianNMT/experiments/dataset']
from google.colab import drive; drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/SicilianNMT-colab'
print('ready')

## 1. The data
Parallel text is the bottleneck for Sicilian. We build it automatically: extract the
*Arba Sicula* PDFs with PyMuPDF, classify each page's language, confirm facing pairs and
align sentences with **LaBSE** embeddings + a monotonic DP (no dictionary, no hunalign);
then add the curated Napizia NLLB / WikiMatrix sets and deduplicate. Code:
`experiments/extraction/build_all.py` and `experiments/dataset/assemble.py` (they need the
PDFs and the public datasets). Here we load the prepared, frozen split from Drive.

In [ ]:
def read(p): return open(p, encoding='utf-8').read().splitlines()
DATA = f'{OUT}/data'
train = {l: read(f'{DATA}/train.{l}') for l in ('scn', 'en')}
test  = {l: read(f'{DATA}/test.{l}')  for l in ('scn', 'en')}
print('train', len(train['scn']), '| test', len(test['scn']))
print('sample pair:\n  scn:', train['scn'][0], '\n  en :', train['en'][0])

## 2. Preprocessing — Standard Sicilian
Sicilian has no single orthography (many spellings, omitted accents). We normalize toward
**Standard Sicilian** (Cipolla's standard) with `normalize_scn.py`: spelling rules like
*cchiù→chiù*, *io/iu/eu→jo*, *non→nun*, and *çiuri→ciuri*. We use the light **std** level
(accents/contractions kept) — our ablation showed it helps NLLB slightly (+0.4 BLEU) while
the aggressive *full* level does not (NLLB's tokenizer wants un-normalized input).

In [ ]:
from normalize_scn import normalize
print('before:', test['scn'][0])
for split in (train, test):
    split['scn'] = [normalize(s, 'std') for s in split['scn']]
print('after :', test['scn'][0])

## 3. The model — NLLB-200 + LoRA
NLLB-200 is a massively multilingual encoder–decoder Transformer that **supports Sicilian**
(`scn_Latn`); it is the equivalent of the costly pretraining stages a from-scratch system
would need. We adapt it with **LoRA**: freeze the weights $W_0$ and learn a low-rank update
$W = W_0 + \frac{\alpha}{r}BA$ ($r{=}32$) on the attention projections — ~1.3% of the
parameters, fits a T4. `load_base` loads NLLB in fp16 (OOM-safe); `attach_lora` adds the
adapter.

In [ ]:
from nllb_pipeline import load_base, attach_lora, build_dataset, finetune, translate, score
model, tok = load_base('facebook/nllb-200-1.3B')
ft = attach_lora(model)

## 4. Fine-tuning — bidirectional scn↔en
We train **both directions at once** (scn→en and en→scn) so one adapter serves both and the
weak reverse direction is rescued. `build_dataset` tags each example with its forced target
language; `finetune` runs the LoRA training (fp16, gradient checkpointing) and saves the
adapter. (~hours on a T4.)

In [ ]:
ds = build_dataset(tok, [
    (train['scn'], train['en'], 'scn', 'en'),
    (train['en'], train['scn'], 'en', 'scn'),
])
print('training examples:', len(ds))
ft = finetune(ft, tok, ds, out_dir=f'{OUT}/nllb-lora-reproduce', epochs=2)

## 5. Evaluation — BLEU & chrF
We score on the **frozen** literary test set with sacreBLEU (`tok:13a`) and chrF, in both
directions. `translate` runs beam search; `score` returns (BLEU, chrF).

In [ ]:
import json
results = {}
for src, tgt in [('scn', 'en'), ('en', 'scn')]:
    hyp = translate(ft, tok, test[src], src, tgt)
    results[f'{src}->{tgt}'] = score(hyp, test[tgt])
    print(f'{src}->{tgt}:  BLEU={results[f"{src}->{tgt}"][0]}  chrF={results[f"{src}->{tgt}"][1]}')
json.dump(results, open(f'{OUT}/results_reproduce.json', 'w'), indent=2)

## 6. Results & relation to the state of the art
Expected (this bidirectional run): **scn→en ≈ 31.4** BLEU, **en→scn ≈ 18.7** — above
Wdowiak's published *baseline* (Sc→En 29.1) on our harder test, but **well below** his
reverse-training state of the art (Sc→En 48.6, En→Sc 45.1, It↔Sc 61–63). We applied the
feasible parts of his recipe on top of NLLB (multilingual, normalization, back-translation)
and each adds little — the residual is **data** (his hand-curated corpus + far more
back-translation), not method. Optional further steps live in their own notebooks:
`sicilian_nllb_multilingual_colab.ipynb` (Italian), `sicilian_nllb_backtranslation_colab.ipynb`.

## 7. Serving
The saved adapter backs a local web app and a Telegram bot — one FastAPI `/translate`
endpoint over `experiments/serving/translator.py`. See `experiments/serving/README.md`:
`uvicorn api:app` serves the site at `/`, and `telegram_bot.py` runs the bot. The bot's
Docker + compose deploy scaffold is in the same folder.